# Apparel Detection Pipeline
Detects clothing items in a video, tracks each item across frames, and outputs a JSON timeline with the first/last/best visible timestamp for each detected apparel.

In [1]:
import cv2
import os
import json
from collections import defaultdict
from PIL import Image
from ultralytics import YOLO

## Configuration

In [ ]:
# ── Input ────────────────────────────────────────────────────────────────────
VIDEO_PATH        = "wedding_ringer.mov"
MODEL_PATH        = "best.pt"         # path to trained YOLO weights

# ── Output ───────────────────────────────────────────────────────────────────
FRAMES_DIR        = "scratch/frames"
CROPS_DIR         = "scratch/apparel_crops"
CROPS_PHASH_DIR   = "scratch/apparel_crops_dedup_phash"
CROPS_CLIP_DIR    = "scratch/apparel_crops_dedup_clip"
OUTPUT_JSON       = "scratch/apparel_timeline.json"

# ── Sampling ─────────────────────────────────────────────────────────────────
SAMPLE_RATE       = 3                 # frames to extract per second

# ── Deduplication ────────────────────────────────────────────────────────────
PHASH_THRESHOLD   = 10                # perceptual hash max distance (lower = stricter)
CLIP_THRESHOLD    = 0.85              # cosine similarity min to call two crops duplicates

In [ ]:
# !rm -rf scratch/

## 1. Extract Frames from Video

In [5]:
os.makedirs(FRAMES_DIR, exist_ok=True)

cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), f"Could not open video: '{VIDEO_PATH}' — check the path"

fps = cap.get(cv2.CAP_PROP_FPS)
frame_interval = int(fps / SAMPLE_RATE)  # how many raw frames to skip between samples

frame_metadata = {}  # maps frame filename -> timestamp in seconds
count = 0
saved = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    if count % frame_interval == 0:
        timestamp = round(count / fps, 3)
        frame_name = f"frame_{saved}.jpg"
        cv2.imwrite(os.path.join(FRAMES_DIR, frame_name), frame)
        frame_metadata[frame_name] = timestamp
        saved += 1

    count += 1

cap.release()
print(f"Extracted {saved} frames at {SAMPLE_RATE} fps (video native: {fps:.1f} fps)")
print(f"Duration covered: {round((count - 1) / fps, 1)}s")

Extracted 108 frames at 3 fps (video native: 60.0 fps)
Duration covered: 35.8s


## 2. Detect Apparel with YOLO

In [ ]:
# Load the trained clothing detection model
model = YOLO(MODEL_PATH)

detections = []  # accumulates all per-frame detections across the video

# Process frames in temporal order
sorted_frames = sorted(frame_metadata.keys(), key=lambda f: frame_metadata[f])

for frame_name in sorted_frames:
    timestamp = frame_metadata[frame_name]
    img_path = os.path.join(FRAMES_DIR, frame_name)
    results = model(img_path)
    print(img_path)
    print(results)

    for r in results:
        boxes = r.boxes.xyxy.cpu().numpy()    # [x1, y1, x2, y2]
        classes = r.boxes.cls.cpu().numpy()   # class index
        confs = r.boxes.conf.cpu().numpy()    # confidence score

        for box, cls, conf in zip(boxes, classes, confs):
            detections.append({
                "frame": frame_name,
                "timestamp": timestamp,
                "box": box.tolist(),
                "class_id": int(cls),
                "class_name": model.names[int(cls)],
                "confidence": float(conf),
            })

print(f"Total detections: {len(detections)}")


image 1/1 /Users/parinshah/Documents/NEU/Sem IV/DL/Project/temporal_fetch/scratch/frames/frame_0.jpg: 416x640 1 short_sleeve_top, 1 long_sleeve_outwear, 78.9ms
Speed: 6.3ms preprocess, 78.9ms inference, 5.4ms postprocess per image at shape (1, 3, 416, 640)

image 1/1 /Users/parinshah/Documents/NEU/Sem IV/DL/Project/temporal_fetch/scratch/frames/frame_1.jpg: 416x640 1 short_sleeve_top, 1 long_sleeve_outwear, 67.2ms
Speed: 2.1ms preprocess, 67.2ms inference, 0.3ms postprocess per image at shape (1, 3, 416, 640)

image 1/1 /Users/parinshah/Documents/NEU/Sem IV/DL/Project/temporal_fetch/scratch/frames/frame_2.jpg: 416x640 1 short_sleeve_top, 1 long_sleeve_outwear, 63.1ms
Speed: 1.4ms preprocess, 63.1ms inference, 0.3ms postprocess per image at shape (1, 3, 416, 640)

image 1/1 /Users/parinshah/Documents/NEU/Sem IV/DL/Project/temporal_fetch/scratch/frames/frame_3.jpg: 416x640 1 short_sleeve_top, 1 long_sleeve_outwear, 58.8ms
Speed: 2.0ms preprocess, 58.8ms inference, 0.3ms postprocess per 

## 3. Track Apparels Across Frames & Save Best Crops
Links the same item across consecutive frames using IoU overlap. For each tracked item, records `first_visible_time`, `last_visible_time`, and `best_visible_time` (the frame with the highest detection confidence). Only the best-frame crop is saved to disk.

In [7]:
os.makedirs(CROPS_DIR, exist_ok=True)

# Tuning knobs
IOU_THRESHOLD = 0.3   # min overlap to link a detection to an existing track
MAX_GAP = 2.0         # seconds of absence before a track is considered ended

def iou(box1, box2):
    """Compute Intersection over Union for two [x1,y1,x2,y2] boxes."""
    x1, y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    x2, y2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

tracks = {}   # track_id -> track info dict
active = {}   # track_id -> most recent detection (used for IoU matching)
next_id = 0

# Group detections by frame for efficient per-frame iteration
by_frame = defaultdict(list)
for det in detections:
    by_frame[det["frame"]].append(det)

for frame_name in sorted(by_frame, key=lambda f: frame_metadata[f]):
    current_time = frame_metadata[frame_name]
    matched = set()

    for det in by_frame[frame_name]:
        best_score, best_tid = 0, None

        # Find the active track of the same class with the highest IoU
        for tid, last in active.items():
            if last["class_id"] != det["class_id"]:
                continue
            if current_time - last["timestamp"] > MAX_GAP:
                continue  # track has gone stale
            score = iou(last["box"], det["box"])
            if score > best_score:
                best_score, best_tid = score, tid

        if best_score >= IOU_THRESHOLD and best_tid not in matched:
            # Continue existing track — update timing and best frame if confidence improved
            track = tracks[best_tid]
            track["last_visible_time"] = current_time
            if det["confidence"] > track["best_confidence"]:
                track["best_visible_time"] = current_time
                track["best_confidence"] = det["confidence"]
                track["best_box"] = det["box"]
                track["best_frame"] = det["frame"]
            active[best_tid] = det
            matched.add(best_tid)
        else:
            # New item — start a fresh track
            tid = next_id
            next_id += 1
            tracks[tid] = {
                "track_id": tid,
                "class_name": det["class_name"],
                "crop_name": f"apparel_{tid}_{det['class_name']}.jpg",
                "first_visible_time": current_time,
                "last_visible_time": current_time,
                "best_visible_time": current_time,
                "best_confidence": det["confidence"],
                "best_box": det["box"],
                "best_frame": det["frame"],
            }
            active[tid] = det
            matched.add(tid)

# Save one crop per track — taken from the highest-confidence frame
for track in tracks.values():
    img = Image.open(os.path.join(FRAMES_DIR, track["best_frame"]))
    x1, y1, x2, y2 = map(int, track["best_box"])
    img.crop((x1, y1, x2, y2)).save(os.path.join(CROPS_DIR, track["crop_name"]))

print(f"Tracked {len(tracks)} apparel items → crops saved to '{CROPS_DIR}'")

Tracked 43 apparel items → crops saved to 'scratch/apparel_crops'


## 4. Export Apparel Timeline to JSON

In [8]:
def to_hhmmss(seconds):
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02}:{m:02}:{s:02}"

output = []

for track in tracks.values():
    output.append({
        "item_id": track["track_id"],
        "class": track["class_name"],
        "crop": track["crop_name"],
        "first_visible_time": to_hhmmss(track["first_visible_time"]),
        "last_visible_time": to_hhmmss(track["last_visible_time"]),
        "best_visible_time": to_hhmmss(track["best_visible_time"]),
    })

# Sort entries by order of first appearance in the video
output.sort(key=lambda x: x["first_visible_time"])

with open(OUTPUT_JSON, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved {len(output)} items to {OUTPUT_JSON}")
print(json.dumps(output[:3], indent=2))  # preview first 3 entries

Saved 43 items to scratch/apparel_timeline.json
[
  {
    "item_id": 0,
    "class": "long_sleeve_outwear",
    "crop": "apparel_0_long_sleeve_outwear.jpg",
    "first_visible_time": "00:00:00",
    "last_visible_time": "00:00:03",
    "best_visible_time": "00:00:01"
  },
  {
    "item_id": 1,
    "class": "short_sleeve_top",
    "crop": "apparel_1_short_sleeve_top.jpg",
    "first_visible_time": "00:00:00",
    "last_visible_time": "00:00:01",
    "best_visible_time": "00:00:00"
  },
  {
    "item_id": 2,
    "class": "long_sleeve_top",
    "crop": "apparel_2_long_sleeve_top.jpg",
    "first_visible_time": "00:00:02",
    "last_visible_time": "00:00:02",
    "best_visible_time": "00:00:02"
  }
]


## 5. Deduplication — Option A: Perceptual Hashing
Fast pixel-level deduplication. Compares crops using a perceptual hash — images that look visually identical will have a hash distance near 0. Processes crops in descending confidence order so the best-quality image is always kept.

> Requires: `pip install imagehash`

In [ ]:
# !pip install imagehash

import imagehash
import shutil

os.makedirs(CROPS_PHASH_DIR, exist_ok=True)

seen_hashes = []  # list of (hash, class_name) — only compare within the same class
kept_phash = []

# Process highest-confidence tracks first so the best image wins when there's a tie
sorted_tracks = sorted(tracks.values(), key=lambda t: -t["best_confidence"])

for track in sorted_tracks:
    crop_path = os.path.join(CROPS_DIR, track["crop_name"])
    h = imagehash.phash(Image.open(crop_path))

    # Only compare against hashes of the same apparel class
    is_duplicate = any(
        abs(h - prev_hash) <= PHASH_THRESHOLD
        for prev_hash, prev_class in seen_hashes
        if prev_class == track["class_name"]
    )

    if not is_duplicate:
        seen_hashes.append((h, track["class_name"]))
        kept_phash.append(track)
        shutil.copy(crop_path, os.path.join(CROPS_PHASH_DIR, track["crop_name"]))

print(f"Perceptual hash: kept {len(kept_phash)} / {len(tracks)} items → '{CROPS_PHASH_DIR}'")

Perceptual hash: kept 31 / 32 items → 'scratch/apparel_crops_dedup_phash'


## 5. Deduplication — Option B: CLIP Embeddings
Semantic deduplication using CLIP vision embeddings. More robust than perceptual hashing — catches the same garment under different lighting, angles, or slight crops. Two items are considered duplicates if their cosine similarity exceeds `CLIP_THRESHOLD`.

> Requires: `pip install transformers torch`

In [ ]:
import torch
import numpy as np
import shutil
from transformers import CLIPProcessor, CLIPModel
from sklearn.metrics.pairwise import cosine_similarity

os.makedirs(CROPS_CLIP_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def get_embedding(image_path):
    """Return a normalized CLIP embedding for a crop image."""
    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        vision_outputs = clip_model.vision_model(pixel_values=inputs["pixel_values"])
        pooled = vision_outputs.pooler_output                    # (1, 768)
        features = clip_model.visual_projection(pooled)          # (1, 512)
    features = features.cpu().numpy()
    return features / np.linalg.norm(features)  # L2 normalize for cosine similarity

# seen_embeddings holds (embedding, class_name) for already-kept crops
seen_embeddings = []
kept_clip = []

sorted_tracks = sorted(tracks.values(), key=lambda t: -t["best_confidence"])

for track in sorted_tracks:
    crop_path = os.path.join(CROPS_DIR, track["crop_name"])
    emb = get_embedding(crop_path)

    # Only compare against embeddings of the same apparel class
    same_class_embs = [e for e, c in seen_embeddings if c == track["class_name"]]

    is_duplicate = False
    if same_class_embs:
        sims = cosine_similarity(emb, np.vstack(same_class_embs))[0]
        is_duplicate = sims.max() >= CLIP_THRESHOLD

    if not is_duplicate:
        seen_embeddings.append((emb, track["class_name"]))
        kept_clip.append(track)
        shutil.copy(crop_path, os.path.join(CROPS_CLIP_DIR, track["crop_name"]))

print(f"CLIP embeddings: kept {len(kept_clip)} / {len(tracks)} items → '{CROPS_CLIP_DIR}'")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP embeddings: kept 19 / 32 items → 'scratch/apparel_crops_dedup_clip'
